# Reproducing Burman et al. (2023) on DANDI:000458: Neuropixels claims

**Paper.** Burman RJ, Brodersen PJN, Raimondo JV, Sen A, Akerman CJ (2023). Active cortical networks
promote shunting fast synaptic inhibition in vivo. *Neuron* 111, 3531-3540.
[doi:10.1016/j.neuron.2023.08.005](https://doi.org/10.1016/j.neuron.2023.08.005)

**Dataset.** [DANDI:000458](https://dandiarchive.org/dandiset/000458/0.230317.0039), version
0.230317.0039 (doi:10.48324/dandi.000458/0.230317.0039), the Allen Institute recordings of Claar et
al. (2023) that the paper reuses (Figure 4A to 4D, Figure S6). This is the only published version
of the dandiset, and the one the paper's analysis code names in its docstring
(gist a77bcf5e, revision 43a13d89), so it is pinned. The draft has the same 24 assets (checked below).

**Text the claims were written from.** The version of record (Europe PMC, PMC11913778), with STAR
Methods; the supplement (Figures S1 to S7) was not available (claims file, opening comment).

**What this notebook tests.** The claims on the dandiset. The network model claims are in
`network_model.ipynb`. The patch-clamp claims (Figures 1 to 3, S1 to S5) rest on recordings that
are not in the dandiset nor deposited anywhere ("available on request"), and are blocked.

| Claim | Here |
|---|---|
| dataset-sessions-link | tested |
| fig4c-sample | tested |
| fig4d-synchrony-lower-awake | tested |
| fig4d-entropy-higher-awake | tested |
| fig4-rate-higher-awake | tested |
| figs6-rate-regressed | tested (regression readings R1 to R3) |
| fig4f-model-hyperpolarizing-less-activity, fig4h-model-synchrony, fig4h-model-entropy, figs7-model-decoding | tested in `network_model.ipynb` |
| figs4-intrinsic-properties | blocked: other (the Figure S4 measure is in the unsupplied supplement; Matlab analysis) |
| 24 patch-clamp claims (fig1b to fig3j, rm-*, figs2, figs3, figs5) | blocked: data_missing |

Every method is the claims file's (opening comment N1 to N8); where it lacks a detail, the passage
of the paper used is cited in the cell or in `npx.py`.

In [ ]:
import time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
import npx

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()

## Data check

The archive's listing and metadata first, then every file the paper's code reads.

In [ ]:
assets = npx.assets()
assets["session"] = assets.path.str.split("/").str[1].str.replace("_behavior+ecephys.nwb", "", regex=False)
meta = npx.version_metadata()
print("version", meta["version"], "| name:", meta["name"])
print("related resources:", [r.get("identifier") or r.get("url") for r in meta.get("relatedResource", [])])
print("assets:", len(assets), "| total size %.1f GB" % (assets["size"].sum() / 1e9))
import requests
draft = requests.get(f"{npx.API}/dandisets/{npx.DANDISET}/versions/draft/assets/?page_size=200", timeout=60).json()["results"]
same = sorted((a["path"], a["size"]) for a in draft) == sorted(zip(assets.path, assets["size"]))
print("draft assets identical in path and size to", npx.VERSION, ":", same)
print("the 17 sessions the gist reads are all assets:", set(npx.SESSIONS) <= set(assets.session))
print("assets not read by the paper's code:", sorted(set(assets.session) - set(npx.SESSIONS)))

In [ ]:
t = time.time()
S_all = {}
for _, a in assets.sort_values("session").iterrows():
    S_all[a.session] = npx.read_session(a.asset_id)
S = {k: S_all[k] for k in npx.SESSIONS}
print(f"loaded {len(S_all)} files in {time.time() - t:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
rows = []
for k, s in S_all.items():
    m = s["meta"]
    if not m["has_units"]:
        rows.append(dict(session=k, in_gist=k in npx.SESSIONS, units=0, note="no units table; acquisition " + ",".join(m["acquisition"])))
        continue
    tr, u = s["trials"], s["units"]
    el = tr[tr.stimulus_type == "electrical"]
    rows.append(dict(
        session=k, in_gist=k in npx.SESSIONS, units=len(u), target=",".join(npx.target_regions(tr)),
        depth=",".join(sorted(el.estim_target_depth.unique())),
        aw_valid=int((el.is_valid & (el.behavioral_epoch == "awake")).sum()),
        iso_valid=int((el.is_valid & (el.behavioral_epoch == "isoflurane")).sum()),
        invalid=int((~el.is_valid.astype(bool)).sum()),
        currents=",".join(sorted(el.estim_current.unique(), key=lambda x: float(x))),
        pulse_ms="%.2f-%.2f" % ((el.stop_time - el.start_time).min() * 1e3, (el.stop_time - el.start_time).max() * 1e3),
        ids_are_rows=m["ids_are_rows"], sorted=m["spike_times_sorted"]))
info = pd.DataFrame(rows)
print(info.to_string())
print("\nsubjects among the 17:", len({k.split('_')[0] for k in npx.SESSIONS}))

The paper's 17 recordings in 16 mice are the 17 files the gist lists (sub-569073 has two sessions,
one stimulating MOs and one SSp). Seven other files are in the version: six hold only EEG and no
units, and one, sub-599017, has units and SSp stimulation but is not in the gist's list; the paper
does not say why it was left out. Each electrical trial lasts 0.4 ms (a biphasic pulse), so
`stop_time` is the pulse's end and `start_time` its onset.

In [ ]:
# Units, variables and ranges used. waveform_duration is described in the Allen data as ms.
U = pd.concat([s["units"].assign(session=k) for k, s in S.items()], ignore_index=True)
print("units in the 17 files:", len(U))
print("waveform_duration range %.3f to %.3f (ms); values exactly 0.4: %d; between 0.399 and 0.401: %d"
      % (U.waveform_duration.min(), U.waveform_duration.max(), (U.waveform_duration == 0.4).sum(),
         U.waveform_duration.between(0.399, 0.401).sum()))
print("distinct waveform_duration steps near 0.4:", np.unique(np.round(U.waveform_duration[U.waveform_duration.between(0.35, 0.45)], 4)))
print("unit locations in stimulated areas (first three characters among estim_target_region):")
for k, s in S.items():
    u = s["units"]; reg = npx.target_regions(s["trials"])
    print("  ", k, reg, u.location[u.location.str[:3].isin(reg)].value_counts().to_dict())
# spike and trial clocks: ranges in seconds
for k, s in list(S.items())[:17]:
    tr = s["trials"]; ep = s["meta"]["epochs"]
    iso = tr[(tr.behavioral_epoch == "isoflurane") & (tr.stimulus_type == "electrical")]
    an = ep[ep.tags.str.contains("anesthesia")]
    inside = ((iso.start_time.values[:, None] >= an.start.values) & (iso.start_time.values[:, None] < an.stop.values)).any(1).mean()
    print(f"{k}: trials {tr.start_time.min():.0f} to {tr.stop_time.max():.0f} s; isoflurane trials inside the "
          f"isoflurane_anesthesia epoch: {inside:.0%}; epochs {list(ep.tags)}")

The isoflurane trials fall inside the file's `isoflurane_anesthesia` epoch, which supports the
`behavioral_epoch` labels. Unit ids equal row numbers in all 17 files, so the gist's keying of
units by row index and lookup by id (claims file, checks, last point) select the same units here.
No unit has a waveform duration of exactly 0.4 ms (the values come in steps of about 0.0137 ms,
0.398 and 0.412 on either side), so the Methods' cut (>= 0.4) and the gist's (> 0.4) select the
same units.

**Clock check.** If the spike times share the trials' clock, spiking 2 to 12 ms after each pulse
should exceed spiking in the same window at times shifted away from the pulses, and differ between
awake and isoflurane trials.

In [ ]:
# Population spike count per trial in 2-12 ms after stop_time, against the same window placed
# 1.5 s after each pulse (inter-pulse intervals are 3.5 to 4.5 s), from the full spike trains of
# the selected units (reading A1). The shifted windows need spikes away from the pulses, read below for two files..
clock = []
for k, s in S.items():
    rows_ = npx.select_units(s, "A1", "ge")
    for c in npx.CONDITIONS:
        tr = npx.condition_trials(s["trials"], c)
        u, t_, x = npx.peristimulus(s, rows_, tr)
        clock.append(dict(session=k, condition=c, trials=len(tr), units=len(rows_),
                          spikes_per_trial_true=len(x) / len(tr)))
clock = pd.DataFrame(clock)
# shifted windows need spikes away from the pulses, which read_session does not keep: read them
# for the units selected, from two files
shift = []
for k in npx.SESSIONS[:2]:
    aid = assets.set_index("session").loc[k, "asset_id"]
    f = npx.open_asset(aid)
    s = S[k]; rows_ = npx.select_units(s, "A1", "ge")
    idx = f["units/spike_times_index"][:]
    st0 = np.r_[0, idx[:-1]]
    for c in npx.CONDITIONS:
        tr = npx.condition_trials(s["trials"], c)
        stops = s["trials"].stop_time.to_numpy()[tr]
        for lag in (0.0, 1.5, -1.5):
            n = 0
            for r in rows_:
                sp = f["units/spike_times"][st0[r]:idx[r]]
                a = np.searchsorted(sp, stops + lag + 0.002); b = np.searchsorted(sp, stops + lag + 0.012)
                n += int((b - a).sum())
            shift.append(dict(session=k, condition=c, lag_s=lag, spikes_per_trial=n / len(tr)))
    f.close()
print(clock.pivot(index="session", columns="condition", values="spikes_per_trial_true").round(2).to_string())
print()
print(pd.DataFrame(shift).pivot_table(index=["session", "condition"], columns="lag_s", values="spikes_per_trial").round(2).to_string())

In both files read at full length, the 2 to 12 ms window after the pulse holds 3 to 18 times the
spikes of the same window 1.5 s before or after it, in both conditions, and the counts differ
between conditions (awake about twice isoflurane). This is consistent with spikes and trials sharing one clock; it is shown on
two of the 17 files, and the evoked counts of the others (table above) are consistent with it.

**Duplicates.** sub-569073 has two files; they are distinct recordings if their trials and spike
counts differ.

In [ ]:
sig = []
for k, s in S.items():
    sig.append(dict(session=k, units=len(s["units"]), spikes=int(s["units"].n_spikes.sum()),
                    first_trial=round(float(s["trials"].start_time.iloc[0]), 5), trials=len(s["trials"])))
sig = pd.DataFrame(sig)
print(sig.to_string())
print("duplicate (units, spikes) pairs:", int(sig.duplicated(["units", "spikes"]).sum()),
      "| duplicate first-trial times:", int(sig.duplicated(["first_trial"]).sum()))

No two files share unit and spike counts or trial times, so each recording is counted once. The same unit in the awake and isoflurane epochs is the same row of one file's units table, sorted over the whole session, so no matching across sessions is needed.

## dataset-sessions-link

> The 17 sessions the paper's analysis code reads are in DANDI:000458 and carry the fields the
> code uses.

In [ ]:
need_units = {"spike_times", "location", "waveform_duration"}
need_trials = {"stimulus_type", "behavioral_epoch", "is_valid", "estim_target_region", "stop_time"}
link = []
for k in npx.SESSIONS:
    s = S[k]
    el = s["trials"][s["trials"].stimulus_type == "electrical"]
    link.append(dict(session=k, file=k in set(assets.session),
                     units_fields=need_units <= set(s["units"].columns) | {"spike_times"},
                     trials_fields=need_trials <= set(s["trials"].columns),
                     awake=(el.behavioral_epoch == "awake").any(),
                     isoflurane=(el.behavioral_epoch == "isoflurane").any()))
link = pd.DataFrame(link)
print(link.to_string())
print("all 17 present with every field and both epochs:", bool(link.drop(columns="session").all().all()))

**Result: reproduced.** All 17 files are in version 0.230317.0039, each with the units fields (spike_times read through its index, location, waveform_duration) and the trial fields the code uses, and with awake and isoflurane electrical trials.

## fig4c-sample

> After excluding 203 neurons with evoked rates below 0.1 Hz or above 200 Hz, 662 regular spiking
> neurons remain, from 17 recordings in 16 mice, a median of 34 per recording (1 to 93).

Judged under the Methods reading A1F1 (the spike count rate); A1F2, A2F1, A2F2 and the gist's
waveform cut are variants (claims file, N7).

In [ ]:
tables, kept = {}, {}
for name, (area, wf, rate) in npx.READINGS.items():
    T = npx.all_units(S, area, wf)
    ex = npx.excluded(T, rate)
    tables[name], kept[name] = T, T[~ex].copy()
rows = []
for name, (area, wf, rate) in npx.READINGS.items():
    T, K = tables[name], kept[name]
    per = K.groupby("session").size().reindex(npx.SESSIONS, fill_value=0)
    low = ((T[f"rate_{rate}_aw"] < 0.1) | (T[f"rate_{rate}_an"] < 0.1)).sum()
    high = ((T[f"rate_{rate}_aw"] > 200) | (T[f"rate_{rate}_an"] > 200)).sum()
    rows.append(dict(reading=name, selected=len(T), kept=len(K), excluded=len(T) - len(K),
                     below_0_1Hz=int(low), above_200Hz=int(high), recordings=int((per > 0).sum()),
                     mice=K.subject.nunique(), median=float(per[per > 0].median()),
                     min=int(per[per > 0].min()), max=int(per.max())))
sample = pd.DataFrame(rows).set_index("reading")
print(sample.to_string())
print("\nexpected: kept 662 +- 66.2, excluded 203 +- 20.3, recordings 17, mice 16, median 34 +- 3.4, min 1 +- 0.1, max 93 +- 9.3")
per = pd.DataFrame({n: kept[n].groupby("session").size().reindex(npx.SESSIONS, fill_value=0) for n in ("A1F1", "A1F2")})
per["selected"] = tables["A1F1"].groupby("session").size().reindex(npx.SESSIONS, fill_value=0)
print("\nper recording:\n" + per.to_string())

In [ ]:
# What the F2 rate is: the gist sums spike times (in ms after the stimulus) where a count is
# expected, so F2 is the mean spike latency (in ms) times F1. Its 200 Hz bound is then a bound
# of 200 Hz divided by the mean latency on the spike-count rate.
T = tables["A1F1"]
ratio = (T.rate_F2_aw / T.rate_F1_aw).replace([np.inf], np.nan)
print("F2 / F1 (awake), median %.2f, 5th to 95th percentile %.2f to %.2f" % (ratio.median(), *ratio.quantile([.05, .95])))
K2 = kept["A1F2"]
print("F1 rate of the units F2 keeps: awake max %.1f Hz, isoflurane max %.1f Hz" % (K2.rate_F1_aw.max(), K2.rate_F1_an.max()))
ex_only_F2 = npx.excluded(T, "F2") & ~npx.excluded(T, "F1")
print("excluded under F2 but not F1:", int(ex_only_F2.sum()), "| under F1 but not F2:", int((npx.excluded(T, "F1") & ~npx.excluded(T, "F2")).sum()))

**Result: partially reproduced.** Under the Methods reading A1F1, 865 units are selected, the
paper's total (662 + 203), and 721 are kept and 144 excluded: the kept count is within tolerance of
662 but the excluded count is outside 203 +- 20.3. Recordings (17) and mice (16) match; the median
per recording is 37 (range 7 to 95). Under the gist's rate (A1F2, and A2F2, which selects the same
units here), every number matches exactly: 662 kept, 203 excluded (143 below 0.1 Hz, 61 above
200 Hz, one unit both), median 34, range 1 to 93. The gist's rate sums spike latencies (F2 / F1
median 4.50), so its 200 Hz bound acts on spike-count rates near 45 Hz (the kept units reach
54.2 Hz); the paper's exclusion matches this rate, not the Methods' count rate.

## fig4d-synchrony-lower-awake

> Neurons show lower stimulus-evoked synchrony with the population awake than under isoflurane
> (0.325 against 0.393).

In [ ]:
def stat_table(measure, readings=npx.READINGS):
    out = []
    for name in readings:
        r = npx.paired(kept[name], measure)
        out.append(dict(reading=name, **{k: (f"{v:.2e}" if k == "p" else round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}))
    return pd.DataFrame(out).set_index("reading")
syn = stat_table("synchrony")
print(syn.to_string())
print("\nexpected awake 0.325 +- 0.005 (0.320 to 0.330), isoflurane 0.393 +- 0.007 (0.386 to 0.400)")
r = npx.paired(kept["A1F1"], "synchrony")
print("A1F1 unrounded: awake %.5f, isoflurane %.5f, p = %.2e" % (r["awake"], r["anesthetized"], r["p"]))

In [ ]:
# Variants: (a) the synchrony population restricted to the kept units (claims file, N6);
# (b) times from start_time, the pulse onset (the Methods say only "following an electrical
# stimulus"; the gist uses stop_time), for A1F1.
def kept_population(reading):
    area, wf, rate = npx.READINGS[reading]
    keep = {(r.session, r.row) for r in kept[reading].itertuples()}
    def pop(session, rows, c, _name=[None]):
        name = [k for k, v in S.items() if v is session][0]
        return np.array([r for r in rows if (name, r) in keep], dtype=int)
    return pop
var = []
for name in ("A1F1", "A1F2"):
    area, wf, rate = npx.READINGS[name]
    T = npx.all_units(S, area, wf, population=kept_population(name))
    K = T.merge(kept[name][["session", "row"]], on=["session", "row"])
    var.append(dict(variant=f"{name}, population = kept units", **npx.paired(K, "synchrony")))
Tst = npx.all_units(S, "A1", "ge", offset=-0.4)
Kst = Tst[~npx.excluded(Tst, "F1")]
var.append(dict(variant="A1F1, times from start_time", **npx.paired(Kst, "synchrony")))
var = pd.DataFrame(var).round(4)
var["p"] = var.p.map(lambda p: f"{p:.2e}")
print(var.to_string())

In [ ]:
# By recording (A1F1), and the range of the plotted values
K = kept["A1F1"]
by = K.groupby("session").apply(lambda d: pd.Series(dict(n=len(d), awake=d.synchrony_aw.mean(),
        isoflurane=d.synchrony_an.mean(), lower_awake=(d.synchrony_aw < d.synchrony_an).mean())), include_groups=False)
print(by.round(3).to_string())
print("recordings with awake mean lower: %d of %d" % ((by.awake < by.isoflurane).sum(), len(by)))
print("units with awake lower: %d of %d" % ((K.synchrony_aw < K.synchrony_an).sum(), len(K)))
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, m, lab in zip(axes, ("synchrony", "entropy"), ("Synchrony", "Entropy (nats)")):
    ax.scatter(K[m + "_an"], K[m + "_aw"], s=4)
    lim = [np.nanmin(K[[m + "_an", m + "_aw"]].values), np.nanmax(K[[m + "_an", m + "_aw"]].values)]
    ax.plot(lim, lim, "k--", lw=1); ax.set_xlabel(lab + ", isoflurane"); ax.set_ylabel(lab + ", awake")
    print(f"{m} plotted range: {lim[0]:.3f} to {lim[1]:.3f}; all points inside the axes")
fig.suptitle("A1F1 kept units"); fig.tight_layout(); plt.show()

**Result: partially reproduced.** Under A1F1 (721 units) synchrony is lower awake, 0.3188 against
0.3862 (paired t test p = 1.45e-32), in the paper's direction and at its significance, but the
awake mean is just outside 0.325 +- 0.005; the isoflurane mean is inside its tolerance, at its
lower edge. Awake is lower in 15 of 17 recordings. Under the gist's rate (A1F2, A2F2) the means
are 0.3253 and 0.3928, the paper's values. Restricting the population to kept units (0.3225
against 0.3876) or timing from pulse onset (0.3073 against 0.3720) keeps the direction.

## fig4d-entropy-higher-awake

> Peri-stimulus histogram entropy is higher awake than under isoflurane (1.347 against 1.111
> nats), so responses are more variable in time.

In [ ]:
ent = stat_table("entropy")
print(ent.to_string())
print("\nexpected awake 1.347 +- 0.018 (1.329 to 1.365), isoflurane 1.111 +- 0.020 (1.091 to 1.131)")
# units left out: entropy is undefined with fewer than two spikes in a condition (N5, N8)
K = kept["A1F1"]
print("A1F1 kept units with fewer than two spikes awake / isoflurane:", int((K.count_aw < 2).sum()), "/", int((K.count_an < 2).sum()))
# means over all units with a value in each condition separately (pandas skipna), a variant
for name in ("A1F1", "A1F2"):
    K = kept[name]
    print(f"{name}, each condition over its own defined values: awake {K.entropy_aw.mean():.3f} (n = {K.entropy_aw.notna().sum()}), "
          f"isoflurane {K.entropy_an.mean():.3f} (n = {K.entropy_an.notna().sum()})")
Kst = Tst[~npx.excluded(Tst, "F1")]
print("A1F1 from start_time:", {k: (f"{v:.2e}" if k == "p" else round(float(v), 4)) for k, v in npx.paired(Kst, "entropy").items()})
by = kept["A1F1"].groupby("session").apply(lambda d: pd.Series(dict(n=d.entropy_aw.notna().sum(),
        awake=d.entropy_aw.mean(), isoflurane=d.entropy_an.mean())), include_groups=False)
print(by.round(3).to_string())
print("recordings with awake mean higher: %d of %d" % ((by.awake > by.isoflurane).sum(), by.awake.notna().sum()))

**Result: partially reproduced.** Under A1F1 entropy is higher awake, 1.389 against 1.150 nats
over 656 units with a value in both conditions (65 dropped with fewer than two spikes in a
condition; paired t test p below 0.001, table above), in the paper's direction and at its
significance, but both means are outside tolerance. Awake is higher in 15 of 17 recordings. Under
the gist's rate (A1F2, A2F2) they are 1.347 and 1.111 nats over 597 units, the paper's values.

## fig4-rate-higher-awake

> The same electrical stimuli evoke more spiking awake than under isoflurane; the mean evoked
> firing rate is higher in the awake state.

In [ ]:
rows = []
for name, (area, wf, rate) in npx.READINGS.items():
    K = kept[name]
    r = npx.paired(K, f"rate_{rate}")
    rows.append(dict(reading=name, rate=rate, **r))
    if rate != "F1":
        rows.append(dict(reading=name, rate="F1 (spike count)", **npx.paired(K, "rate_F1")))
rt = pd.DataFrame(rows).round(3)
rt["p"] = rt.p.map(lambda p: f"{p:.2e}")
print(rt.to_string())
K = kept["A1F1"]
print("units with a higher awake rate: %d of %d" % ((K.rate_F1_aw > K.rate_F1_an).sum(), len(K)))
by = K.groupby("session")[["rate_F1_aw", "rate_F1_an"]].mean()
print("recordings with higher awake mean rate: %d of %d" % ((by.rate_F1_aw > by.rate_F1_an).sum(), len(by)))
print(by.round(2).to_string())

**Result: reproduced.** Under A1F1 the mean evoked rate is 14.0 Hz awake against 7.59 Hz under
isoflurane (paired t test, t = 12.36, p below 0.05, table above), higher awake in 539 of 721 units
and 15 of 17 recordings; every reading gives the same direction.

## figs6-rate-regressed

> The awake decrease in synchrony and increase in entropy remain after the effect of firing rate
> is regressed out.

Readings from the criterion: (R1) one linear regression of the measure on the evoked rate across
all unit and condition values, residuals compared by the paired t test; (R2) OLS of the measure on
condition and rate, condition coefficient tested; (R3) per unit, the awake minus isoflurane
difference of the measure regressed on the difference of rate, intercept tested. The rate is the
reading's own (F1 under A1F1).

In [ ]:
def regressed(K, measure, rate):
    a, b = K[measure + "_aw"], K[measure + "_an"]
    ok = a.notna() & b.notna()
    K = K[ok]
    long = pd.DataFrame({"y": np.r_[K[measure + "_aw"], K[measure + "_an"]],
                         "rate": np.r_[K[f"rate_{rate}_aw"], K[f"rate_{rate}_an"]],
                         "awake": np.r_[np.ones(len(K)), np.zeros(len(K))]})
    # R1
    slope, icpt = np.polyfit(long.rate, long.y, 1)
    res = long.y - (icpt + slope * long.rate)
    n = len(K)
    t1 = stats.ttest_rel(res[:n], res[n:])
    # R2
    m2 = smf.ols("y ~ awake + rate", data=long).fit()
    # R3
    d = pd.DataFrame({"dy": K[measure + "_aw"].values - K[measure + "_an"].values,
                      "dr": K[f"rate_{rate}_aw"].values - K[f"rate_{rate}_an"].values})
    m3 = smf.ols("dy ~ dr", data=d).fit()
    return dict(n=n, R1_diff=res[:n].mean() - res[n:].mean(), R1_p=t1.pvalue,
                R2_awake_coef=m2.params["awake"], R2_p=m2.pvalues["awake"],
                R3_intercept=m3.params["Intercept"], R3_p=m3.pvalues["Intercept"])
rows = []
for name, (area, wf, rate) in npx.READINGS.items():
    for m in ("synchrony", "entropy"):
        rows.append(dict(reading=name, measure=m, **regressed(kept[name], m, rate)))
reg = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.3g}".format):
    print(reg.to_string())
print("\nexpected signs: synchrony awake difference negative, entropy positive, p < 0.05 under every regression reading")
for m, sign in (("synchrony", -1), ("entropy", 1)):
    r = reg[(reg.reading == "A1F1") & (reg.measure == m)].iloc[0]
    ok = all(np.sign(r[k]) == sign and r[p] < 0.05 for k, p in (("R1_diff", "R1_p"), ("R2_awake_coef", "R2_p"), ("R3_intercept", "R3_p")))
    print(f"A1F1 {m}: holds under R1, R2 and R3: {ok}")

In [ ]:
# By recording, R2 under A1F1 (a breakdown; recordings with fewer than 5 units are skipped)
rows = []
for k, d in kept["A1F1"].groupby("session"):
    if len(d) < 5:
        continue
    for m in ("synchrony", "entropy"):
        r = regressed(d, m, "F1")
        rows.append(dict(session=k, measure=m, n=r["n"], R2_awake_coef=r["R2_awake_coef"], R2_p=r["R2_p"]))
br = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.3g}".format):
    print(br.pivot(index="session", columns="measure", values=["R2_awake_coef", "R2_p"]).to_string())
for m, sign in (("synchrony", -1), ("entropy", 1)):
    d = br[br.measure == m]
    print(f"{m}: awake coefficient in the expected direction in {int((np.sign(d.R2_awake_coef) == sign).sum())} of {len(d)} recordings")
print(f"\nnotebook run time so far {(time.time() - T0) / 60:.1f} min")

**Result: reproduced.** Under A1F1, with the evoked rate regressed out, synchrony stays lower and
entropy higher awake under each of R1, R2 and R3 (all six p below 1e-10, table above), and the
same holds under every other reading, so the outcome does not depend on which regression the
unread Figure S6 legend describes.

## Summary

| Claim | Outcome |
|---|---|
| dataset-sessions-link | reproduced |
| fig4c-sample | partially reproduced (exact under the gist's rate) |
| fig4d-synchrony-lower-awake | partially reproduced (exact under the gist's rate) |
| fig4d-entropy-higher-awake | partially reproduced (exact under the gist's rate) |
| fig4-rate-higher-awake | reproduced |
| figs6-rate-regressed | reproduced (under all three regression readings) |
| model claims (four) | see `network_model.ipynb` |
| figs4-intrinsic-properties | blocked: other (the Figure S4 measure is in the unsupplied supplement; Matlab analysis) |
| 24 patch-clamp claims | blocked: data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The claims file judges each claim under the
Methods' evoked rate (spike count over trials times 10 ms). The paper's numbers come out exactly
under the rate its own code computes, which sums the spikes' latencies where a count is expected;
under that rate the exclusion drops units above about 45 Hz of spike-count rate. The three partial
outcomes are this difference: the effects are there, at the paper's significance, with means a
little off under the Methods' rate.

**What was found about the dataset.** The 17 recordings the paper used are in the pinned version
with every field needed, and their labels and clocks pass the checks above. The version also holds
sub-599017 (units, SSp stimulation), which the paper does not use and does not mention, and six
EEG-only files. The patch-clamp recordings of Figures 1 to 3 are not in the dandiset.